# Gen 1 / OS 2 field-test starter

For a new checkout, copy this template to test.ipynb and edit the hostname. Run the steps in order. test.ipynb is ignored by Git so your settings and results stay local.

## 1. Select the board and image

Edit the hostname. This cell only checks which local PyRPL installation and fork bitstream will be used.

In [ ]:
import json
from pathlib import Path
import sys

import pyrpl
from pyrpl.redpitaya import RedPitaya

HOSTNAME = "rp-xxxxxx.local"
SSH_USER = "root"
bitstream = Path(pyrpl.__file__).resolve().parent / "fpga" / "red_pitaya.bin"

print("Python:", sys.executable)
print("PyRPL:", pyrpl.__file__)
print("Bitstream:", bitstream)
assert bitstream.is_file()

## 2. Run the read-only preflight

Enter the SSH password. This checks the OS, board profile, overlay contract, and local FPGA files without uploading or programming anything.

In [ ]:
from getpass import getpass

SSH_PASSWORD = getpass("SSH password: ")
device = None
try:
    device = RedPitaya(
        config=None,
        hostname=HOSTNAME,
        user=SSH_USER,
        password=SSH_PASSWORD,
        gui=False,
        autostart=False,
        reloadfpga=False,
        reloadserver=False,
    )
    preflight_report = device.preflight_fpga_update(filename=str(bitstream))
finally:
    if device is not None:
        device.end_ssh()

print(json.dumps(preflight_report, indent=2, sort_keys=True))

## 3. Program the FPGA

This uploads the fork BIN and matching DTBO and programs the FPGA. It does not start the PyRPL server. Use only on the intended original Z7010 board.

In [ ]:
device = None
try:
    device = RedPitaya(
        config=None,
        hostname=HOSTNAME,
        user=SSH_USER,
        password=SSH_PASSWORD,
        filename=str(bitstream),
        gui=False,
        autostart=False,
        reloadfpga=False,
        reloadserver=False,
    )
    program_report = device.update_fpga(filename=str(bitstream))
finally:
    if device is not None:
        device.end_ssh()

print(json.dumps(program_report, indent=2, sort_keys=True))

## 4. Start PyRPL and connect

This installs/starts the monitor server and checks the fork register map. It does not reprogram the FPGA.

In [ ]:
from pyrpl import Pyrpl

p = Pyrpl(
    config="gen1-os2-field-test",
    hostname=HOSTNAME,
    user=SSH_USER,
    password=SSH_PASSWORD,
    filename=str(bitstream),
    gui=False,
    reloadfpga=False,
    reloadserver=True,
)
rp = p.rp
print("PyRPL connected; fork register checks passed.")

## 5. Read a module setting

This reads PID0's input-filter setting without driving an output. Further analog/control checks require their own measurements.

In [ ]:
print("PID0 input filter:", rp.pid0.inputfilter)